# Diffusion 1d

In [1]:
%load_ext autoreload
%autoreload 2
%matplotlib widget
from src.Diffusion1d import *
from src.trainloop import *
import numpy as np
import pandas as pd
import joblib
from src.DataConfig import DataConfig
from src.ConditioningDropout import CondDropout
import tempfile, torch
from src.DataPipeline import DataPipeline, make_splits, add_share_cols, hash_checker
from ema_pytorch import EMA
import wandb
wandb.login()
device = "cuda" if torch.cuda.is_available() else "cpu"

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\Samuel\_netrc.
wandb: Currently logged in as: natusvictus (natusvictus-fh-aachen-university-of-applied-sciences) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


## Load Data and set Config Params

In [2]:
# Data Locations
target_root = Path(r"../data/split")

split_dirs = {
    "train": target_root/"train",
    "val": target_root/"val",
    "test": target_root/"test",
    "holdout_building": target_root/"building_holdout",
    "holdout_year": target_root/"year_holdout",
    "holdout_station": target_root/"station_holdout",

}
#make sure cleaned is used everytime since otherwise the apl cols are missing silently
MANIFEST_DIR = Path(r"../data/manifests") / "cleaned"
manifests = load_split_manifests({
    "train": MANIFEST_DIR/"manifest_train_realised_cleaned.csv",
    "val":   MANIFEST_DIR/"manifest_val_realised_cleaned.csv",
    "test":  MANIFEST_DIR/"manifest_test_realised_cleaned.csv",
    "holdout_building": MANIFEST_DIR/"manifest_holdout_building_realised_cleaned.csv",
    "holdout_year": MANIFEST_DIR/"manifest_holdout_year_realised_cleaned.csv",
    "holdout_station": MANIFEST_DIR/"manifest_holdout_station_realised_cleaned.csv",
})

train_df = manifests["train"]
strat_cols = ["buildingType", "buildingAgeBin"]
# all dynamic columns for stat caclculation
STATS_COLS =  ["Electricity Load", "Hot Water Load","Heating Load","Occupancy Home Active", "Occupancy Home Not Active","T","Off Day"]
# Aggregation Method in case of other than 1min Res
TS_AGG = {
    "T": "first",
    "Occupancy Home Active": "first",
    "Occupancy Home Not Active": "first",
    "Electricity Load": "mean",
    "Hot Water Load": "mean",
    "Heating Load": "mean",
    "Off Day": "max"
}

train: 1500 rows, 1470 buildings
val: 250 rows, 201 buildings
test: 250 rows, 147 buildings
holdout_building: 60 rows, 10 buildings
holdout_year: 150 rows, 97 buildings
holdout_station: 150 rows, 88 buildings


In [3]:
# todo check what happens when no conditioning is used
# Cols to Train on and Generate
TARGET_COLS = ["Electricity Load", "Hot Water Load","Heating Load"] #"T", "Occupancy Home Not Active", "Occupancy Home Not Active","Heating Load", "Electricity Load", "Hot Water Load"

# Timeseries Cols to Condition on
DYNAMIC_COND_COLS = ["T", "Occupancy Home Active", "Occupancy Home Not Active","Off Day"] #"T", "Occupancy Home Active", "Occupancy Home Not Active", "T"

# Context Use i.e. steps of previous window as conditioning
CONTEXT_STEPS = 60
CONTEXT_REVERSE = True
N_CTX = len(TARGET_COLS) + 1 if CONTEXT_STEPS > 0 else 0

# doy as conditioning
USE_SEASON = True
N_SEASON = 2 if USE_SEASON else 0
N_DY_CON_COLS = len(DYNAMIC_COND_COLS) + N_SEASON + N_CTX

# Static Parameters to condition on
STATIC_CAT_COLS = ["buildingType","buildingAgeBin","year_type","climateRegion"]
#"buildingType","buildingAgeBin","year_type","climateRegion"
STATIC_BOOL_COLS = ["future"] #"future"
STATIC_CONT_COLS = ["a_ref","n_apartments","n_persons_sum"] + add_share_cols(train_df)
#"a_ref","n_apartments","n_persons_sum


# Cols for normalization of the Target cols so that the load distribution isnt weird across buildings
# define additional normalization by area/occpancy per domain
NORM_METHOD="none"

norm_variant={
    "none": {},
    "elec_sh_scale": {"Electricity Load": "n_persons_sum","Heating Load": "a_ref"},
    "elec_dhw_sh_scale": {"Electricity Load": "n_persons_sum","Hot Water Load": "n_persons_sum","Heating Load": "a_ref"}
}
NORM_BY_COL = norm_variant[NORM_METHOD]


#Cols where mean is set to zero for scaling
ZERO_COLS = []

# define indivdual scalers per column default is zscore
SCALER_BY_COL = {
    "Electricity Load": "log_zscore",
    "Hot Water Load": "log_zscore",
    "Heating Load": "log_zscore",
    "Off Day": "minmax"
}

# dropout groups and probs
cond_groups = {
    # "occupancy": ["Occupancy Home Active","Occupancy Home Not Active"],
    # "temperature": ["T"],
    # "off_day": ["Off Day"],
    # "season": "season"
}

cond_dropout = {
    # "occupancy": 0.2,
    # "temperature": 0.3,
    # "off_day": 0.1,
    # "season": 0.1
}

static_dropout={
    # "cat": 0.1, "cont": 0.1
}


cond_dropout_all=0.1

In [4]:
STATIC_CONT_COLS

['a_ref',
 'n_apartments',
 'n_persons_sum',
 'apl_ANSWER_MACHINE',
 'apl_CD_PLAYER',
 'apl_CHEST_FREEZER',
 'apl_CLOCK',
 'apl_DISH_WASHER',
 'apl_FAX',
 'apl_FRIDGE',
 'apl_FRIDGE_FREEZER',
 'apl_HIFI',
 'apl_HOB',
 'apl_IRON',
 'apl_KETTLE',
 'apl_MICROWAVE',
 'apl_OVEN',
 'apl_PC',
 'apl_PHONE',
 'apl_PRINTER',
 'apl_RECEIVER',
 'apl_TUMBLE_DRYER',
 'apl_TV1',
 'apl_TV2',
 'apl_TV3',
 'apl_UPRIGHT_FREEZER',
 'apl_VACUUM',
 'apl_VCR_DVD',
 'apl_WASHER_DRYER',
 'apl_WASHING_MACHINE']

In [5]:
# Parameters for Windowing and Dataloader
WINDOW_DAYS = 1
STRIDE_DAYS = WINDOW_DAYS
RESOLUTION = "1min"

STEPS_PER_DAY = stepsperday(RESOLUTION)
WINDOW_LENGTH = WINDOW_DAYS * STEPS_PER_DAY
STRIDE = STRIDE_DAYS * STEPS_PER_DAY

In [6]:
cfg = DataConfig(
    target_cols=TARGET_COLS,
    dynamic_cond_cols=DYNAMIC_COND_COLS,
    static_cat_cols=STATIC_CAT_COLS,
    static_cont_cols=STATIC_CONT_COLS,
    static_bool_cols=STATIC_BOOL_COLS,
    resolution=RESOLUTION,
    window_days=WINDOW_DAYS,
    stride_days=STRIDE_DAYS,
    ts_agg=TS_AGG,
    stats_cols=STATS_COLS,
    norm_method=NORM_METHOD,
    norm_variant=norm_variant,
    scaler_by_col=SCALER_BY_COL,
    context_steps=CONTEXT_STEPS,
    context_reverse=CONTEXT_REVERSE,
    use_season=USE_SEASON,
    strat_cols=strat_cols,
    split_dirs={k: str(v) for k, v in split_dirs.items()},
    cond_groups=cond_groups,
    cond_dropout=cond_dropout,
    cond_dropout_all=cond_dropout_all,
    static_dropout=static_dropout,
    buffer_size=50,
)

In [7]:
cond = CondDropout.from_config(cfg)
pipe = DataPipeline.build(cfg, manifests)
schedule = DiffusionSchedule.make("sigmoid", timesteps=1000).to(device)

preprocessor = pipe.preprocessor
cat_cardinalities = pipe.cat_cardinalities
stats = pipe.stats
target_scaler = pipe.target_scaler
condition_scaler= pipe.condition_scaler
X0_MIN1, X0_MAX1 = pipe.x0_minmax()

train: 1500 rows, 1470 buildings
val: 250 rows, 201 buildings
test: 250 rows, 147 buildings
holdout_building: 60 rows, 10 buildings
holdout_year: 150 rows, 97 buildings
holdout_station: 150 rows, 88 buildings
Missing Files: 0
Existing Files: 1500
Total: 1500
Missing Files: 0
Existing Files: 250
Total: 250
Missing Files: 0
Existing Files: 250
Total: 250
Missing Files: 0
Existing Files: 60
Total: 60
Missing Files: 0
Existing Files: 150
Total: 150
Missing Files: 0
Existing Files: 150
Total: 150


In [8]:
preprocessor.static_cont_cols

['a_ref',
 'n_apartments',
 'n_persons_sum',
 'apl_ANSWER_MACHINE',
 'apl_CD_PLAYER',
 'apl_CHEST_FREEZER',
 'apl_CLOCK',
 'apl_DISH_WASHER',
 'apl_FAX',
 'apl_FRIDGE',
 'apl_FRIDGE_FREEZER',
 'apl_HIFI',
 'apl_HOB',
 'apl_IRON',
 'apl_KETTLE',
 'apl_MICROWAVE',
 'apl_OVEN',
 'apl_PC',
 'apl_PHONE',
 'apl_PRINTER',
 'apl_RECEIVER',
 'apl_TUMBLE_DRYER',
 'apl_TV1',
 'apl_TV2',
 'apl_TV3',
 'apl_UPRIGHT_FREEZER',
 'apl_VACUUM',
 'apl_VCR_DVD',
 'apl_WASHER_DRYER',
 'apl_WASHING_MACHINE']

In [9]:

train_df_exist, val_df_exist, test_df_exist = (pipe.split_dfs[s] for s in ("train", "val", "test"))
train_files, train_run_ids, train_static = (pipe.prepare_data("train")[k] for k in ("files", "run_ids", "static"))
val_files, val_run_ids, val_static = (pipe.prepare_data("val")[k] for k in ("files", "run_ids", "static"))
test_files, test_run_ids, test_static = (pipe.prepare_data("test")[k] for k in ("files", "run_ids", "static"))

train_dataset = pipe.dataset("train", train=True)

train_dataset_eval = pipe.windowdataset("train", pipe.Window_drawer("train", 1, 1, seed=0))
val_dataset_eval = pipe.windowdataset("val", pipe.Window_drawer("val", 1, 1, seed=0))

In [15]:
BATCH_SIZE = 256
train_dataloader = pipe.dataloader(train_dataset, BATCH_SIZE, train=True)

train_eval_dataloader = pipe.dataloader(train_dataset_eval, BATCH_SIZE)
val_eval_dataloader = pipe.dataloader(val_dataset_eval, BATCH_SIZE)

In [16]:
ds = val_eval_dataloader.dataset
print("windows in index:", len(ds))
print("runs/files:", len(ds.files))
print("batch_size:", val_eval_dataloader.batch_size,
      "| drop_last:", val_eval_dataloader.drop_last,
      "| num_workers:", val_eval_dataloader.num_workers)
print("batches yielded:", sum(1 for _ in val_eval_dataloader))

windows in index: 468
runs/files: 39
batch_size: 256 | drop_last: False | num_workers: 4
batches yielded: 4


## Training Loop

In [17]:
EPOCHS = 1
LOSS_TYPE =  "l2"

In [18]:
model_cfg = make_model_cfg(pipe=pipe,
                           model_type="1d",
                           dim=64,
                           dim_mults=(1,2,4),
                           mid_attn="full",
                           level_attn="none",
                           use_day_index=False)

optim_cfg = dict(lr=1e-4)
ema_cfg = dict(beta=0.999, update_after_step=100, update_every=3, power=2/3)

model = build_model(model_cfg, device)
optimizer = build_optimizer(model, optim_cfg)
ema = build_ema(model, ema_cfg)
BOUNDARY_WEIGHT = [4.0, 1.0, 10.0]
config = {
    "epochs": EPOCHS, "timesteps": schedule.timesteps, "schedule": "sigmoid",
    "loss_type": LOSS_TYPE, "lr": optimizer.param_groups[0]["lr"],
    "model": model_cfg, "optimizer": optim_cfg, "ema": ema_cfg,
    "data": cfg.to_dict(), "split_hash": pipe.split_hash, "boundary_weight": BOUNDARY_WEIGHT
}
X0_MIN1, X0_MAX1 = pipe.x0_minmax()
extras = {"x0_min": X0_MIN1, "x0_max": X0_MAX1}
exp_trgt_cols = ''.join(s.lower().replace('load', '').replace(' ', '') for s in cfg.target_cols)
experiment = f"{model_cfg["model_type"]}_{exp_trgt_cols}"
run_dir = make_run_dir(config, root=r"../outputs/runs", experiment=experiment)
pipe.save_files(run_dir / "data")

saved data pipeline (split 07684ff027fc) to ..\outputs\runs\20261120_105721_29d46e_1d_electricityhotwaterheating\data


In [19]:
comment = "Rework_Test"
run = wandb.init(project="diffusion-model_test_runs", name=f"{comment}_{run_dir.name}",group="1d_run_tests", config=config)
train(model,
      schedule,
      train_dataloader,
      optimizer,
      ema,
      device,
      epochs=EPOCHS,
      loss_type=LOSS_TYPE,
      run=run,
      run_dir=run_dir,
      config=config,
      extras=extras,
      val_dataloader=val_eval_dataloader,
      train_eval_dataloader=train_eval_dataloader,
      val_every_steps=5000,
      val_batches=None,
      ckpt_every_steps=5000,
      val_raw=False,
      boundary_weight=BOUNDARY_WEIGHT,
      cond=cond)
run.finish()

epoch,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
global_step,▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
train/learning_rate,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/loss,█▇▆▅▅▄▄▄▄▄▃▂▂▂▃▂▁▁▁▁
epoch,1
global_step,19
train/learning_rate,0.0001
train/loss,0.52993


epoch 1/1: 0it [00:14, ?it/s]

Loss: 1.5851
Loss: 0.2855
Loss: 0.1032
Loss: 0.1231
Loss: 0.1092
Loss: 0.1122
Loss: 0.1342
Loss: 0.0594
Loss: 0.0450
Loss: 0.0684
Loss: 0.0928
Loss: 0.0713
Loss: 0.0387
Loss: 0.0875
Loss: 0.0569
Loss: 0.0730
Loss: 0.0380
Loss: 0.0413
Loss: 0.0409
Loss: 0.0331
Loss: 0.0360
Loss: 0.0861
Epoch 1: loss=0.0955


epoch,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch/loss,▁
global_step,▁▁▁▁▁▂▂▂▂▂▃▃▃▃▃▃▃▄▄▄▅▅▅▅▅▅▅▆▆▆▇▇▇▇▇█████
train/learning_rate,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/loss,█▄▄▄▂▃▃▄▄▂▁▂▂▃▂▂▂▂▁▁▂▂▃▂▂▁▂▃▂▁▁▂▁▂▂▂▂▁▃▁
epoch,1
epoch/loss,0.09549
global_step,2135
train/learning_rate,0.0001
train/loss,0.03782


In [20]:
model.ups

ModuleList(
  (0): ModuleList(
    (0): Sequential(
      (0): Upsample(scale_factor=2.0, mode='nearest')
      (1): Conv1d(256, 128, kernel_size=(3,), stride=(1,), padding=(1,))
    )
    (1): ResnetBlock1d(
      (mlp): Sequential(
        (0): SiLU()
        (1): Linear(in_features=256, out_features=256, bias=True)
      )
      (block1): Block1d(
        (proj): WeightStandardizedConv1d(256, 128, kernel_size=(3,), stride=(1,), padding=(1,))
        (norm): GroupNorm(8, 128, eps=1e-05, affine=True)
        (act): SiLU()
      )
      (block2): Block1d(
        (proj): WeightStandardizedConv1d(128, 128, kernel_size=(3,), stride=(1,), padding=(1,))
        (norm): GroupNorm(8, 128, eps=1e-05, affine=True)
        (act): SiLU()
      )
      (res_conv): Conv1d(256, 128, kernel_size=(1,), stride=(1,))
    )
  )
  (1): ModuleList(
    (0): Sequential(
      (0): Upsample(scale_factor=2.0, mode='nearest')
      (1): Conv1d(128, 64, kernel_size=(3,), stride=(1,), padding=(1,))
    )
    (1